In [ ]:
%pip -q install transformers

from transformers import AutoTokenizer

# Bài 12: Tokenization bằng tokenizer của pretrained model
MODEL_NAME = "bert-base-uncased"
texts = [
    "The cats are playing.",
    "The cats are playing in the garden with their friends.",
]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# padding=True đưa các câu trong batch về cùng độ dài.
# truncation=True cắt câu nếu vượt max_length.
encoded = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt",
)

print("Tokenizer:", MODEL_NAME)
print("Output keys:", list(encoded.keys()))
print("input_ids: các số ID ánh xạ tới token trong vocabulary.")
print("attention_mask: 1 = token hợp lệ, 0 = vị trí padding cần bỏ qua.")

for i, text in enumerate(texts):
    ids = encoded["input_ids"][i].tolist()
    attention_mask = encoded["attention_mask"][i].tolist()
    tokens = tokenizer.convert_ids_to_tokens(ids)
    special_mask = tokenizer.get_special_tokens_mask(
        ids, already_has_special_tokens=True
    )

    print(f"\n--- Câu {i + 1} ---")
    print("Văn bản:", text)
    print("Số từ (tách theo khoảng trắng):", len(text.split()))
    print("Subword pieces:", tokenizer.tokenize(text))
    print("Tokens (gồm special/padding tokens):", tokens)
    print("input_ids:", ids)
    print("attention_mask:", attention_mask)
    print("special token mask (1 = token đặc biệt):", special_mask)

print("\nSpecial tokens:", tokenizer.special_tokens_map)
print("Kích thước batch input_ids:", tuple(encoded["input_ids"].shape))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer: bert-base-uncased
Output keys: ['input_ids', 'token_type_ids', 'attention_mask']
input_ids: các số ID ánh xạ tới token trong vocabulary.
attention_mask: 1 = token hợp lệ, 0 = vị trí padding cần bỏ qua.

--- Câu 1 ---
Văn bản: The cats are playing.
Số từ (tách theo khoảng trắng): 4
Subword pieces: ['the', 'cats', 'are', 'playing', '.']
Tokens (gồm special/padding tokens): ['[CLS]', 'the', 'cats', 'are', 'playing', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
input_ids: [101, 1996, 8870, 2024, 2652, 1012, 102, 0, 0, 0, 0, 0, 0]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
special token mask (1 = token đặc biệt): [1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1]

--- Câu 2 ---
Văn bản: The cats are playing in the garden with their friends.
Số từ (tách theo khoảng trắng): 10
Subword pieces: ['the', 'cats', 'are', 'playing', 'in', 'the', 'garden', 'with', 'their', 'friends', '.']
Tokens (gồm special/padding tokens): ['[CLS]', 'the', 'cats', 'are', 'playing

In [ ]:
# Bài 13: Classification Head trên pretrained BERT
import torch
from torch import nn
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "bert-base-uncased"
LABELS = ["negative", "positive"]
torch.manual_seed(42)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class BertTextClassifier(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        # Classification head là lớp riêng cho task, khởi tạo ngẫu nhiên.
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        encoder_outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        # Lấy contextual representation của [CLS] ở vị trí đầu tiên.
        cls_representation = encoder_outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_representation)
        return logits, cls_representation

model = BertTextClassifier(MODEL_NAME, num_labels=len(LABELS))
model.eval()

text = "The movie was excellent."
inputs = tokenizer(
    text,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt",
)

with torch.no_grad():
    logits, cls_representation = model(**inputs)
    probabilities = torch.softmax(logits, dim=-1)
    predicted_id = probabilities.argmax(dim=-1).item()

print("Text:", text)
print("Input keys:", list(inputs.keys()))
print("[CLS] representation shape:", tuple(cls_representation.shape))
print("Logits:", logits[0].tolist())
print("Softmax probabilities:", probabilities[0].tolist())
print("Demo prediction (head chưa huấn luyện):", LABELS[predicted_id])

# Khi huấn luyện với nhãn thật:
# loss_fn = nn.CrossEntropyLoss()
# loss = loss_fn(logits, labels)
# loss.backward()

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Text: The movie was excellent.
Input keys: ['input_ids', 'token_type_ids', 'attention_mask']
[CLS] representation shape: (1, 768)
Logits: [-0.18573135137557983, -0.3150612711906433]
Softmax probabilities: [0.5322874784469604, 0.46771252155303955]
Demo prediction (head chưa huấn luyện): negative
